In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif

# 1. Load the data
df = pd.read_csv("scaled_data.csv")

# 2. Separate features (X) and target (y)
X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

# 3. Feature Selection: Select the top 15 most relevant features
# KNN suffers from the "curse of dimensionality", so using fewer columns is optimal
selector = SelectKBest(score_func=f_classif, k=15)
X_selected = selector.fit_transform(X, y)

# Let's see which features were selected
selected_columns = X.columns[selector.get_support()]
print("Selected Features:\n", selected_columns.tolist(), "\n")

# 4. Split the data into training and testing sets
# We use 'stratify=y' because bankruptcies are very rare compared to non-bankruptcies
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Create a KNN model and use GridSearchCV to find the optimal hyperparameters
knn = KNeighborsClassifier()

# Define a grid of parameters to test
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# We use ROC-AUC as the scoring metric because of the class imbalance
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Optimal Parameters Found: {grid_search.best_params_}\n")

# 6. Evaluate the optimal model on the test data
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test)
y_pred_proba = best_knn.predict_proba(X_test)[:, 1]

# 7. Print Performance Metrics
print("--- Model Performance ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")